# Set Card Classifier — Deployment with TorchScript (`.pt`)

This notebook demonstrates how to load and run the exported Set Card Classifier for **production inference** — no PyTorch Lightning required.

## Export format: TorchScript vs `torch.export`

Two PyTorch export APIs exist. This project uses **TorchScript** (`torch.jit.trace`):

| | TorchScript `.pt` | `torch.export` `.pt2` |
|---|---|---|
| API | `torch.jit.trace` / `torch.jit.script` | `torch.export.export` |
| Dynamic batch size | Yes, out of the box | Requires `dynamic_shapes` — unreliable across versions |
| Maturity | Production-proven | Newer, still evolving |
| Requires Lightning | No | No |
| File size | ~43 MB | ~43 MB |

`torch.export` bakes the example input's batch size as a **static shape guard**, causing an `AssertionError` for any other batch size. Working around this with `dynamic_shapes` is inconsistent across PyTorch versions. TorchScript traces are batch-size agnostic by design.

## What this notebook covers

1. Loading the model and inspecting its interface
2. Preprocessing — the most common deployment mistake
3. Single-image inference from a **file path**
4. Single-image inference from **raw bytes** (web server pattern)
5. Understanding the full probability output — not just the top-1 prediction
6. Batch inference from **file paths**
7. Batch inference from **pre-built tensors** (production server pattern)
8. Best practices summary

**Run order:** top to bottom. The `preprocess()` and `decode()` helpers defined in Section 2 are reused throughout.

In [ ]:
import os
import sys
import glob
import time
import numpy as np
import torch
import cv2
from pathlib import Path

# Anchor the working directory to the project root.
# This makes all relative paths (data/, checkpoints/) work correctly
# regardless of how Jupyter was launched (VS Code, JupyterLab, etc.).
for _p in [Path.cwd()] + list(Path.cwd().parents):
    if (_p / "src").exists() and (_p / "data").exists():
        os.chdir(_p)
        break

PROJECT_ROOT = Path.cwd()

# Device selection.
# CUDA (NVIDIA GPU) gives the highest throughput for batched inference.
# MPS (Apple Silicon) is a good middle ground on macOS.
# CPU works everywhere and is often fast enough for single-image inference.
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Working directory : {PROJECT_ROOT}")
print(f"PyTorch version   : {torch.__version__}")
print(f"Device            : {DEVICE}")

## 1. Load the Model

Loading a TorchScript model requires only `torch.jit.load()` — no model class definition, no Lightning.

Two things to do after loading:
- **`.to(DEVICE)`** — move weights to the target accelerator
- **`.eval()`** — switch batch norm and dropout to inference mode

**Best practice:** Load the model **once at startup** and reuse it for every request. Loading takes ~1–2 seconds and allocates GPU memory. Reloading per request is a common and costly mistake in production deployments.

In [ ]:
MODEL_PATH = "checkpoints/model.pt"

# torch.jit.load() restores the TorchScript traced model (graph + weights).
# No knowledge of the original model class or PyTorch Lightning required.
# map_location moves weights directly to the target device on load.
model = torch.jit.load(MODEL_PATH, map_location=DEVICE)

# Unlike torch.export models, TorchScript models are NOT in eval mode by default.
# Always call .eval() before inference to ensure batch norm and dropout
# use inference-time behaviour (running statistics, no dropout).
model.eval()

print(f"Loaded : {MODEL_PATH}")
print(f"Device : {DEVICE}")

In [ ]:
# Confirm the model accepts any batch size — the key advantage of TorchScript
# over torch.export for this use case.
for batch_size in [1, 4, 8]:
    dummy = torch.zeros(batch_size, 3, 224, 224, device=DEVICE)
    with torch.no_grad():
        out = model(dummy)
    shapes = {f: tuple(t.shape) for f, t in out.items()}
    print(f"batch={batch_size}  output: { {f: s for f, s in shapes.items()} }")

print()
print("Input  : Tensor(batch_size, 3, 224, 224)")
print("         3 channels = RGB  |  224x224 pixels  |  ImageNet-normalized")
print()
print("Output (one key per Set card feature, any batch size works):")
for feature, tensor in out.items():
    print(f"  '{feature:8s}' -> Tensor(batch_size, 3)  (raw logits, one value per class)")

## 2. Preprocessing — The Most Common Deployment Mistake

The model was trained with a specific preprocessing pipeline. **You must apply the exact same pipeline at inference time.** Getting this wrong is the single most common cause of poor deployment performance — the model won't throw an error, it will silently predict wrong.

The pipeline has four steps, applied in order:

| Step | Operation | Why |
|------|-----------|-----|
| 1 | **Convert BGR → RGB** | OpenCV loads images in BGR order by default |
| 2 | **Resize to 224×224** | ResNet18 was trained at this exact resolution |
| 3 | **Scale to [0.0, 1.0]** | Divide by 255 |
| 4 | **ImageNet normalization** | Subtract per-channel mean, divide by std |

The ImageNet statistics are fixed constants — they come from the ImageNet dataset and apply to **any model pretrained on ImageNet** (ResNet, VGG, EfficientNet, etc.):

```
Mean : [0.485, 0.456, 0.406]   (R, G, B channels)
Std  : [0.229, 0.224, 0.225]   (R, G, B channels)
```

The cell below defines two reusable helpers used throughout this notebook:
- **`preprocess(img_rgb)`** — applies all four steps, returns a model-ready tensor
- **`decode(logits, idx)`** — converts raw logits to human-readable predictions with confidence scores

In [ ]:
# ── Label mappings ────────────────────────────────────────────────────────────
# Must exactly match what was used during training.
# Key: class index (int output from argmax). Value: human-readable label string.
LABELS = {
    "color":   {0: "red",     1: "green",   2: "purple"},
    "shape":   {0: "diamond", 1: "squiggle", 2: "oval"},
    "number":  {0: "1",       1: "2",        2: "3"},
    "shading": {0: "solid",   1: "striped",  2: "open"},
}
FEATURES = list(LABELS.keys())

# ── ImageNet normalization constants ──────────────────────────────────────────
# These are fixed properties of the ImageNet dataset — do not change them.
# They apply to any model whose backbone was pretrained on ImageNet.
_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def preprocess(img_rgb: np.ndarray) -> torch.Tensor:
    """Converts an HxWx3 RGB uint8 image into a model-ready (1, 3, 224, 224) tensor.

    This pipeline must replicate training preprocessing exactly.
    Any deviation (wrong mean/std, BGR instead of RGB, missing resize)
    will produce silently wrong predictions with no error raised.

    Args:
        img_rgb: HxWx3 numpy array, RGB channel order, dtype uint8, values [0, 255].

    Returns:
        Tensor of shape (1, 3, 224, 224) on DEVICE, ready for the model.
    """
    # Step 1: Resize to the resolution the model was trained on.
    # INTER_LINEAR is a good default — fast and accurate for downscaling.
    img = cv2.resize(img_rgb, (224, 224), interpolation=cv2.INTER_LINEAR)

    # Step 2: Scale pixel values from integer [0, 255] to float [0.0, 1.0].
    img = img.astype(np.float32) / 255.0

    # Step 3: Apply per-channel ImageNet normalization: (value - mean) / std.
    # This centers each channel around 0 with unit variance, matching
    # the pixel distribution the pretrained ResNet18 backbone was trained on.
    img = (img - _MEAN) / _STD

    # Step 4: Rearrange from HxWxC (numpy) to CxHxW (PyTorch), add batch dim.
    img = img.transpose(2, 0, 1)                          # (3, 224, 224)
    return torch.from_numpy(img).unsqueeze(0).to(DEVICE)  # (1, 3, 224, 224)


def decode(logits: dict, idx: int = 0) -> dict:
    """Converts raw model logits for one image into human-readable predictions.

    Args:
        logits: Dict of {feature: Tensor(batch_size, 3)} from a forward pass.
        idx:    Index within the batch to decode (default 0 for single images).

    Returns:
        Dict of {feature: {"prediction": str, "confidence": float,
                            "probabilities": {label: float}}}
    """
    result = {}
    for feature in FEATURES:
        # Softmax converts raw logits to a proper probability distribution (sums to 1).
        # Always apply softmax — never compare raw logits as if they were probabilities.
        probs     = torch.softmax(logits[feature][idx], dim=0)  # (3,)
        class_idx = probs.argmax().item()
        result[feature] = {
            "prediction": LABELS[feature][class_idx],
            "confidence":  round(probs[class_idx].item(), 4),
            # Return the full distribution — callers can threshold on confidence
            # or detect ambiguous predictions (e.g. 51% vs 49%).
            "probabilities": {
                LABELS[feature][i]: round(probs[i].item(), 4)
                for i in range(3)
            },
        }
    return result


print("Helpers defined.")
print(f"Features : {FEATURES}")
print(f"Classes  : { {f: list(v.values()) for f, v in LABELS.items()} }")

## 3. Single-Image Inference — From a File Path

The simplest pattern: load an image from disk and classify it.

Use this for:
- CLI tools and batch processing scripts
- Local evaluation and debugging
- Any workflow with direct filesystem access

**Key reminder:** OpenCV's `cv2.imread()` always loads images in **BGR** order. Convert to RGB before calling `preprocess()`. This is the single most common mistake when deploying CV models.

In [ ]:
IMAGE_PATH = "data/raw/red_diamond_1_solid.jpg"

# ── Load ──────────────────────────────────────────────────────────────────────
img_bgr = cv2.imread(IMAGE_PATH)                        # HxWx3, BGR, uint8
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)      # HxWx3, RGB, uint8

# ── Preprocess ────────────────────────────────────────────────────────────────
tensor = preprocess(img_rgb)                            # (1, 3, 224, 224)

# ── Inference ─────────────────────────────────────────────────────────────────
# torch.no_grad() disables gradient tracking — always use it for inference.
# Skipping it does not affect the output but wastes memory and slows things down.
with torch.no_grad():
    logits = model(tensor)                              # dict: feature -> Tensor(1, 3)

# ── Decode ────────────────────────────────────────────────────────────────────
result = decode(logits)

print(f"Image : {IMAGE_PATH}\n")
for feature, info in result.items():
    print(f"  {feature:8s}: {info['prediction']:10s}  confidence: {info['confidence']:.1%}")

## 4. Single-Image Inference — From Raw Bytes

In a web server (Flask, FastAPI, etc.), images arrive as **raw bytes** over HTTP — not as file paths. This is the real deployment pattern.

The key is `cv2.imdecode()`: it decodes a compressed image (JPEG, PNG, etc.) directly from a byte array into a numpy array — no temporary file on disk needed.

```python
# FastAPI example (not run here):
@app.post("/predict")
async def predict(file: UploadFile):
    image_bytes = await file.read()
    nparr   = np.frombuffer(image_bytes, dtype=np.uint8)
    img_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    tensor  = preprocess(img_rgb)
    with torch.no_grad():
        logits = model(tensor)
    return decode(logits)
```

The cell below simulates this by reading the same image file as raw bytes first, then treating those bytes as if they arrived over the network.

In [ ]:
# Simulate receiving image bytes from an HTTP request.
# In production:
#   image_bytes = await request.body()   # FastAPI
#   image_bytes = request.data           # Flask
with open(IMAGE_PATH, "rb") as f:
    image_bytes = f.read()

print(f"Received {len(image_bytes):,} bytes\n")

# ── Decode bytes -> numpy (no temp file needed) ───────────────────────────────
# np.frombuffer interprets the raw bytes as a 1-D uint8 array.
# cv2.imdecode then decompresses that buffer as a JPEG/PNG/etc. image.
nparr   = np.frombuffer(image_bytes, dtype=np.uint8)
img_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)        # HxWx3, BGR
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)      # HxWx3, RGB

# ── Preprocess -> Inference -> Decode (identical to file-path approach) ────────
tensor = preprocess(img_rgb)
with torch.no_grad():
    logits = model(tensor)
result = decode(logits)

print("Result (from raw bytes):\n")
for feature, info in result.items():
    print(f"  {feature:8s}: {info['prediction']:10s}  confidence: {info['confidence']:.1%}")

## 5. Understanding the Output — Probabilities, Not Just Predictions

The model outputs **raw logits** for each feature — one value per class. After softmax, these become a probability distribution over the three classes.

**Always surface the full distribution in production — not just the top-1 label.** Here is why:

- A prediction with **99% confidence** is very different from **51% confidence**, even if the top-1 label is the same.
- Low-confidence predictions signal **ambiguous input** (poor lighting, partial occlusion, wrong preprocessing) and should be flagged for human review.
- Logging the full distribution lets you **monitor model drift** over time — watch for the average confidence dropping across a feature.

A reasonable production threshold: flag any prediction below **80% confidence** for review.

In [ ]:
# Show the full probability distribution for the last result.
CONFIDENCE_THRESHOLD = 0.80

print("Full probability distribution:\n")
for feature, info in result.items():
    print(f"  {feature}:")
    sorted_probs = sorted(info["probabilities"].items(), key=lambda x: -x[1])
    for label, prob in sorted_probs:
        bar      = "█" * int(prob * 30)
        is_pred  = label == info["prediction"]
        marker   = " <- predicted" if is_pred else ""
        print(f"    {label:10s}  {prob:5.1%}  {bar}{marker}")
    print()

# Flag low-confidence predictions
low_confidence = [
    f"{feature}: {info['prediction']} ({info['confidence']:.1%})"
    for feature, info in result.items()
    if info["confidence"] < CONFIDENCE_THRESHOLD
]
if low_confidence:
    print(f"WARNING: low-confidence predictions (< {CONFIDENCE_THRESHOLD:.0%}):")
    for item in low_confidence:
        print(f"  {item}")
else:
    print(f"All predictions above {CONFIDENCE_THRESHOLD:.0%} confidence threshold.")

## 6. Batch Inference — From File Paths

**Batching is the single most impactful throughput optimization.** Instead of calling the model N times (one image per call), you stack all images into a single `(N, 3, 224, 224)` tensor and run **one forward pass**.

Why batching is faster:
- **GPU parallelism** — the GPU processes all N images simultaneously across its thousands of cores
- **Kernel launch overhead** — one GPU kernel launch vs N launches
- **Weight cache** — model weights are loaded into cache once, not N times

The cell below runs both approaches on the same images and prints the speedup so you can see the difference on your hardware.

In [ ]:
image_paths = glob.glob("data/raw/*.jpg")[:8]

# ── Approach 1: naive loop (one forward pass per image) ───────────────────────
t0 = time.perf_counter()
for path in image_paths:
    img_bgr = cv2.imread(path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    tensor  = preprocess(img_rgb)
    with torch.no_grad():
        _ = model(tensor)
loop_ms = (time.perf_counter() - t0) * 1000

# ── Approach 2: batch (preprocess all, stack, one forward pass) ───────────────
imgs = [
    preprocess(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)).squeeze(0)
    for p in image_paths
]                                               # list of (3, 224, 224) tensors
batch_tensor = torch.stack(imgs)                # (N, 3, 224, 224)

t0 = time.perf_counter()
with torch.no_grad():
    _ = model(batch_tensor)
batch_ms = (time.perf_counter() - t0) * 1000

n = len(image_paths)
print(f"N = {n} images")
print(f"  Loop  ({n} forward passes) : {loop_ms:6.1f} ms   ({loop_ms/n:.1f} ms/image)")
print(f"  Batch (1 forward pass)   : {batch_ms:6.1f} ms   ({batch_ms/n:.1f} ms/image)")
print(f"  Speedup                  : {loop_ms/batch_ms:.1f}x")

In [ ]:
# ── Batch inference with decoded results ──────────────────────────────────────
# .squeeze(0) removes the batch dimension added by preprocess() so we can
# re-stack them ourselves into one (N, 3, 224, 224) tensor.
imgs = [
    preprocess(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)).squeeze(0)
    for p in image_paths
]
batch_tensor = torch.stack(imgs)                # (N, 3, 224, 224)

with torch.no_grad():
    logits = model(batch_tensor)                # each value: Tensor(N, 3)

print(f"Input batch shape : {tuple(batch_tensor.shape)}")
print(f"Output per feature: Tensor(N=8, 3)\n")
print("Predictions:")
for i, path in enumerate(image_paths):
    result = decode(logits, idx=i)
    preds = " | ".join(
        f"{f}: {r['prediction']} ({r['confidence']:.0%})"
        for f, r in result.items()
    )
    print(f"  [{i}] {os.path.basename(path)}")
    print(f"       {preds}")

## 7. Batch Inference — From Pre-Built Tensors

In a real server, images rarely come from disk. They arrive as a **stream of byte payloads** — HTTP request bodies, message queue messages, etc. The server accumulates them into a batch before calling the model.

This is the most realistic production pattern:

```
Request 1 ──┐
Request 2 ──┤  collect N requests
   ...      ├──────────────────────> preprocess each -> stack -> model -> unpack
Request N ──┘
```

**Why collect into a batch?** Because the GPU is underutilized by single-image requests. Collecting a small batch (4–32 images) before firing a forward pass dramatically improves throughput without meaningfully increasing latency.

The cell below simulates this end-to-end: N byte payloads arrive, get decoded and preprocessed individually, then merged into one tensor for a single forward pass.

In [ ]:
BATCH_SIZE = 4

# ── Simulate N images arriving as raw bytes ───────────────────────────────────
# In production these come from HTTP request bodies, S3, a queue, etc.
byte_payloads = []
for path in image_paths[:BATCH_SIZE]:
    with open(path, "rb") as f:
        byte_payloads.append(f.read())

print(f"Received {len(byte_payloads)} byte payloads")
print(f"Payload sizes: {[len(b) for b in byte_payloads]} bytes\n")

# ── Steps 1-3: decode each payload and preprocess into a tensor ───────────────
# Each image is handled independently here. In a real server, these steps
# could run in parallel (ThreadPoolExecutor) since they are CPU-bound.
tensors = []
for payload in byte_payloads:
    nparr   = np.frombuffer(payload, dtype=np.uint8)   # raw bytes -> 1-D uint8 array
    img_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)    # decompress JPEG/PNG -> HxWx3 BGR
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)  # BGR -> RGB
    tensors.append(preprocess(img_rgb).squeeze(0))      # -> (3, 224, 224)

# ── Step 4: stack individual tensors into one batch ───────────────────────────
# torch.stack requires all tensors to have the same shape.
batch = torch.stack(tensors)                            # (BATCH_SIZE, 3, 224, 224)
print(f"Batch tensor shape : {tuple(batch.shape)}")

# ── Step 5: single forward pass ───────────────────────────────────────────────
with torch.no_grad():
    logits = model(batch)                               # dict: feature -> Tensor(N, 3)

# ── Step 6: unpack — one result dict per input image ─────────────────────────
print("\nResults (one per input payload):\n")
for i in range(BATCH_SIZE):
    result = decode(logits, idx=i)
    print(f"  Image {i} ({os.path.basename(image_paths[i])})")
    for feature, info in result.items():
        print(f"    {feature:8s}: {info['prediction']:10s}  {info['confidence']:.1%}")
    print()

## 8. Best Practices Summary

### Model Loading
| Practice | Why |
|---|---|
| Load **once at startup**, reuse across requests | Loading takes 1–2 s and allocates GPU memory |
| Call `.to(DEVICE)` immediately after loading | All input tensors must be on the same device |
| Skip `.eval()` for `.pt2` | Eval mode is baked in at export time |

### Preprocessing
| Practice | Why |
|---|---|
| **RGB, not BGR** | OpenCV loads BGR — always convert before `preprocess()` |
| Use the **exact same pipeline as training** | Any deviation silently degrades accuracy |
| Use `cv2.imdecode()` for byte inputs | Avoids writing temp files to disk |
| Always wrap inference in `torch.no_grad()` | Disables gradient tracking — reduces memory, improves speed |

### Output
| Practice | Why |
|---|---|
| Return the **full probability distribution** | Callers can threshold, detect ambiguity, and monitor drift |
| Flag predictions **below ~80% confidence** | Signals poor image quality or out-of-distribution input |
| Never compare raw logits as probabilities | Always apply softmax first |

### Throughput
| Practice | Why |
|---|---|
| **Batch when possible** (4–32 images) | Maximizes GPU utilization |
| Keep tensors **on the GPU** | CPU↔GPU transfers are often the bottleneck |
| Parallelize **preprocessing** (CPU-bound) | `ThreadPoolExecutor` for concurrent `cv2.imdecode` + `preprocess` calls |
| **Profile before optimizing** | On CPU/MPS the bottleneck is preprocessing; on GPU it is data transfer |